# Cold Start Chat
## Chat system for addressing the cold start problem head on.
This will be using NLP to take in a user's input and parse any relevant information to then add to the users profile, e.g. they like horror films, they don't like Brad Pitt, etc.
This is to give them an immediate set of recommendations based on their profiles.

Another objective in these questions is to not ask redundant ones, e.g. if they've already said they don't like non-English films, we shouldn't ask them if they like French films.


### Libraries
- Recommendation library: LensKit
- NLP library: spaCy

In [1]:
import numpy as np
import lenskit
import pandas as pd
import spacy

print("NumPy version:", np.__version__)
print("LensKit version:", lenskit.__version__)
print("Pandas version:", pd.__version__)
print("spaCy version:", spacy.__version__)

NumPy version: 2.0.2
LensKit version: 0.14.4
Pandas version: 2.2.3
spaCy version: 3.8.2


/Users/Cathal/Rec-Genie/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
nlp = spacy.load('en_core_web_sm')

user_profile = {
    'films_liked': [],
    'films_disliked': [],
    'genres_positive': [],
    'genres_negative': [],
    'actors_positive': [],
    'actors_negative': [],
    'directors_positive': [],
    'directors_negative': [],
    'languages_positive': [],
    'languages_negative': []
}

In [16]:
# Initialize an empty user profile
def initialize_user_profile():
    user_profile = {
        'films_liked': [],
        'films_disliked': [],
        'genres_positive': [],
        'genres_negative': [],
        'actors_positive': [],
        'actors_negative': [],
        'directors_positive': [],
        'directors_negative': [],
        'languages_positive': [],
        'languages_negative': []
    }

In [ ]:
# Define a list of directors, actors, genres, and languages from the dataset


In [13]:
import spacy
from transformers import pipeline

# Load spaCy model
nlp = spacy.load('en_core_web_sm')
from transformers import pipeline, AutoTokenizer, TFAutoModelForSequenceClassification
# Load Twitter-roBERTa-base for Sentiment Analysis
tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment")
model = TFAutoModelForSequenceClassification.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment")
sentiment_analyzer = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)


def parse_user_input(user_input):
    """
    Parse user input for relevant information about likes and dislikes.
    """
    doc = nlp(user_input)
    entities = {'films': [], 'genres': [], 'actors': [], 'directors': [], 'languages': []}
    
    # Extract entities using spaCy's NER
    # for ent in doc.ents:
    #     if ent.label_ == 'PERSON':
    #         entities['actors'].append(ent.text)
    #     elif ent.label_ == 'WORK_OF_ART':
    #         entities['films'].append(ent.text)
    #     elif ent.label_ == 'LANGUAGE':
    #         entities['languages'].append(ent.text)
        
        # Add more entity types as needed
    
    # # Analyze sentiment for the entire sentence
    # sentences = [sent.text for sent in doc.sents]
    # for sentence in sentences:
    #     sentiment = sentiment_analyzer(sentence)[0]
    #     for key in entities.keys():
    #         for i, entity in enumerate(entities[key]):
    #             if entity in sentence:
    #                 if sentiment['label'] == 'NEGATIVE':
    #                     entities[key][i] = (entity, 'dislike')
    #                 elif sentiment['label'] == 'POSITIVE':
    #                     entities[key][i] = (entity, 'like')

    # Analyze sentiment all entities
    for key in entities.keys():
        for i, entity in enumerate(entities[key]):
            sentiment = sentiment_analyzer(entity[0])[0]
            if sentiment['label'] == 'NEGATIVE':
                entities[key][i] = (False, entity)
            elif sentiment['label'] == 'POSITIVE':
                entities[key][i] = (True, 'like')

    return entities

All model checkpoint layers were used when initializing TFRobertaForSequenceClassification.

All the layers of TFRobertaForSequenceClassification were initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFRobertaForSequenceClassification for predictions without further training.


In [14]:
def update_user_profile(user_profile, parsed_data):
    """
    Update the user profile based on parsed data.
    """
    for key in parsed_data.keys():
        for entity in parsed_data[key]:
            print(entity)
            
            # user_profile[key + '_' + entity + 'd'].append(entity[0][0])
            
    return user_profile

user_input = "I like horror films. I like bullet train. I hate Brad Pitt. I hate French films. I like English films."


# Parse user input
parsed_data = parse_user_input(user_input)

initialize_user_profile()
# Update the user profile
user_profile = update_user_profile(user_profile, parsed_data)

print("Updated User Profile:")
print(user_profile)
                

Brad Pitt
English
Updated User Profile:
{'films_liked': [], 'films_disliked': [], 'genres_positive': [], 'genres_negative': [], 'actors_positive': [], 'actors_negative': [], 'directors_positive': [], 'directors_negative': [], 'languages_positive': [], 'languages_negative': []}


In [15]:
# Sample user input
# user_input = "I love horror films and Sandra Bullock, but I can't stand Brad Pitt. I prefer movies in English. I don't like Chinese films. I don't like Brad Pitt.I don't like French films."
user_input = "I hate Brad Pitt"


# Parse user input
parsed_data = parse_user_input(user_input)
print(parsed_data)

initialize_user_profile()
# Update the user profile
user_profile = update_user_profile(user_profile, parsed_data)

print("Updated User Profile:")
print(user_profile)


{'films': [], 'genres': [], 'actors': ['Brad Pitt'], 'directors': [], 'languages': []}
Brad Pitt
Updated User Profile:
{'films_liked': [], 'films_disliked': [], 'genres_positive': [], 'genres_negative': [], 'actors_positive': [], 'actors_negative': [], 'directors_positive': [], 'directors_negative': [], 'languages_positive': [], 'languages_negative': []}


In [6]:
# Sample user input
# user_input = "I love horror films and Sandra Bullock, but I can't stand Brad Pitt. I prefer movies in English. I don't like Chinese films. I don't like Brad Pitt.I don't like French films."
user_input = "I don't like Sandra Bullock"


# Parse user input
parsed_data = parse_user_input(user_input)

# Update the user profile
user_profile = update_user_profile(user_profile, parsed_data)

print("Updated User Profile:")
print(user_profile)


Updated User Profile:
{'films_liked': [], 'films_disliked': [], 'genres_positive': [], 'genres_negative': [], 'actors_positive': [], 'actors_negative': [], 'directors_positive': [], 'directors_negative': [], 'languages_positive': [], 'languages_negative': []}


In [7]:
# Sample questions to ask the user
questions = [
    "Have you seen any good movies recently?",
    "What are your favorite genres?",
    "Who are your favorite actors?",
    "Who are your favorite directors?",
    "Do you have any favorite languages for films?",
    "Are there any actors you dislike?",
    "Are there any directors you dislike?",
    "Are there any genres you dislike?",
    "Are there any languages you dislike?"
]

# Ask the user questions
for question in questions:
    user_input = input(question + " ")
    parsed_data = parse_user_input(user_input)
    user_profile = update_user_profile(user_profile, parsed_data)
    
print("Final User Profile:")
print(user_profile)





KeyboardInterrupt: Interrupted by user